# 🎙️ Narrador con tu propia voz (Chatterbox Multilingual, español)

**Creado por Eduardo Rincón** · [github.com/neoworld09/Voz-IA](https://github.com/neoworld09/Voz-IA) · Licencia CC BY-NC 4.0 (puedes usarlo y compartirlo dando crédito; no se permite venderlo).

Genera la narración de un video corto (Shorts/TikTok/Reels) frase por frase con **tu propia voz**, gratis en Google Colab.

1. **Entorno de ejecución → Cambiar tipo de entorno → GPU T4** → Guardar.
2. **Entorno de ejecución → Ejecutar todo**. La primera vez, al terminar de instalar, la sesión **se reinicia sola**: espera unos segundos y dale **Ejecutar todo otra vez**.
3. En la celda 2 sube un audio tuyo (WAV/MP3, 10–30 s, sin música ni ruido).
4. Ajusta EMOCION y RITMO con la celda 5 y genera todo con la celda 6.

⚠️ **Uso responsable:** usa solo tu propia voz o la de alguien que te haya dado permiso explícito. No clones voces de otras personas ni las uses para engañar o suplantar.

Modelo: [Chatterbox Multilingual](https://github.com/resemble-ai/chatterbox) de Resemble AI (licencia MIT). Los audios generados llevan una marca de agua inaudible (Perth).

In [ ]:
#@title 1 · Instalar (tarda 2–4 min la primera vez)
import importlib.util, os
if importlib.util.find_spec('chatterbox') is None:
    !pip install -q chatterbox-tts==0.1.7 2>&1 | grep -v -i 'dependency resolver\|requires\|incompatible' || true
    !pip uninstall -y -q torchvision
    print('✅ Instalado. La sesión se reinicia sola: espera 5 s y dale Ejecutar todo OTRA VEZ.')
    os.kill(os.getpid(), 9)
else:
    print('Ya está instalado ✅')

In [ ]:
#@title 2 · Sube tu voz de referencia (10–30 s, solo tu voz, sin música)
from google.colab import files
import subprocess, os
subido = files.upload()
nombre = next(iter(subido))
subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', nombre, '-ac', '1', '-ar', '24000', 'mi_voz.wav'], check=True)
print('✅ Voz de referencia lista: mi_voz.wav')

In [ ]:
#@title 3 · Guion (edita las frases: una por línea de audio)
FRASES = [
 "Esta es la primera frase de tu video.",
 "Y esta es la segunda. Cambia estas frases por tu guion.",
 "¿Te gustó? Cuéntame en los comentarios.",
]

In [ ]:
#@title 4 · Cargar el modelo (solo la primera vez, ~1 min)
import torch, torchaudio as ta, zipfile, shutil
from chatterbox.mtl_tts import ChatterboxMultilingualTTS
from google.colab import files
from IPython.display import Audio, display
if 'model' not in globals():
    dev = 'cuda' if torch.cuda.is_available() else 'cpu'
    print('Usando', dev)
    model = ChatterboxMultilingualTTS.from_pretrained(device=dev)
print('Modelo listo ✅')

## 🎧 Prueba rápida
Mueve **EMOCION** y **RITMO** aquí abajo y dale ▶️ **solo a esta celda**. En unos segundos escuchas 2 frases de muestra. Repite hasta que te guste y luego dale ▶️ a la celda 6.

In [ ]:
#@title 5 · Escuchar una muestra (unos segundos)
EMOCION = 0.8 #@param {type:'slider', min:0.25, max:1.0, step:0.05}
RITMO = 0.6 #@param {type:'slider', min:0.2, max:0.8, step:0.05}
FRASE_DE_PRUEBA = 1 #@param {type:'integer'}
for k in sorted({0, max(0, min(len(FRASES)-1, FRASE_DE_PRUEBA-1))}):
    w = model.generate(FRASES[k], language_id='es', audio_prompt_path='mi_voz.wav', exaggeration=EMOCION, cfg_weight=RITMO)
    ta.save(f'prueba_{k}.wav', w, model.sr)
    print(f'Frase {k+1}:', FRASES[k])
    display(Audio(f'prueba_{k}.wav'))
print(f'EMOCION={EMOCION}  RITMO={RITMO}')

In [ ]:
#@title 6 · Generar TODO con esos ajustes y descargar
shutil.rmtree('salida', ignore_errors=True); os.makedirs('salida')
for i, t in enumerate(FRASES):
    wav = model.generate(t, language_id='es', audio_prompt_path='mi_voz.wav', exaggeration=EMOCION, cfg_weight=RITMO)
    ta.save(f'salida/linea_{i:02d}.wav', wav, model.sr)
    print(f'{i+1}/{len(FRASES)} ✅', t[:60])
with zipfile.ZipFile('narracion.zip','w') as z:
    for f in sorted(os.listdir('salida')): z.write('salida/'+f, f)
print('Listo. Si no se descarga solo: panel 📁 a la izquierda → narracion.zip → ⋮ → Descargar')
files.download('narracion.zip')